Montreal Cycling Infrastructure: Data to Map

This notebook reads Montreal's bicycle-network GeoJSON, keeps the records in the City of Montreal, renames selected fields for the web map, writes a cleaned GeoJSON file, and checks that output.



1. Read the source data 

The source is the City of Montreal's Réseau cyclable dataset, published through Données Québec under CC BY 4.0. View the dataset. Accessed October 1, 2026.
The source GeoJSON contains the geometry and attributes for bicycle-network segments. The source values, including French facility names, will stay unchanged; this notebook renames selected property keys into English.

In [1]:
import json

with open("../data/bike_lanes.geojson") as f:
    data = json.load(f)
print(type(data))
print(data.keys())
print(len(data["features"]))
print(data["features"][0]["properties"])

<class 'dict'>
dict_keys(['type', 'crs', 'features'])
9485
{'ID_CYCL': 27608, 'ID_TRC': 0.0, 'AFFICHEUR_DYNAMIQUE': 'Non', 'AVANCEMENT_CODE': 'E', 'AVANCEMENT_DESC': 'Voie cyclable existante', 'COMPTEUR_CYLISTE': 'Non', 'LONGUEUR': 326, 'NBR_VOIE': 2, 'NOM_ARR_VILLE_CODE': 'KL', 'NOM_ARR_VILLE_DESC': 'Kirkland', 'PROTEGE_4S': 'Non', 'REV_AVANCEMENT_CODE': None, 'REV_AVANCEMENT_DESC': None, 'ROUTE_VERTE': 'Non', 'SAISONS4': 'Non', 'SAS_VELO': 'Non', 'SEPARATEUR_CODE': None, 'SEPARATEUR_DESC': None, 'TYPE_VOIE_CODE': '5', 'TYPE_VOIE_DESC': 'Piste cyclable en site propre', 'TYPE_VOIE2_CODE': '44', 'TYPE_VOIE2_DESC': 'Piste bidirectionnelle', 'VILLE_MTL': 'Non'}


 2. Filter and clean data

Keep only features where VILLE_MTL is Oui. This focuses the map on the City of Montreal rather than off-island linked municipalities. 
Remove features with missing geometry or blank values in the required fields: lane type, protection status, length, and area. 
Count both types of exclusions and prints so the cleaning is clear.
Builds a new feature with the original geometry and six selected properties: the five used by the map and the source ID_TRC field, retained for joining a related street-segment table. Renames the map property keys into English but leaves the original values unchanged.
Three cleaned records have no ID_TRC value and 682 have ID_TRC set to 0; check those records against the pavement table before joining.
Writes the cleaned features to ../data/bike_lanes_clean_english.geojson.


In [2]:
import json

with open("../data/bike_lanes.geojson") as f:
    raw = json.load(f)

clean_features = []
dropped_outside_city = 0
dropped_incomplete = 0

for feature in raw["features"]:
    props = feature.get("properties", {})

    # keep only Montreal
    if props.get("VILLE_MTL") != "Oui":
        dropped_outside_city += 1
        continue

    # remove missing geometry
    geometry = feature.get("geometry")
    if geometry is None:
        dropped_incomplete += 1
        continue

    # remove blank required fields
    required_fields = ["TYPE_VOIE_DESC", "PROTEGE_4S", "LONGUEUR", "NOM_ARR_VILLE_DESC"]
    has_blank = False

    for field in required_fields:
        value = props.get(field)
        if value is None or str(value).strip() == "":
            has_blank = True
            break

    if has_blank:
        dropped_incomplete += 1
        continue

    # rename fields to English
    translated_feature = {
        "type": "Feature",
        "geometry": geometry,
        "properties": {
            "facility_type": props.get("TYPE_VOIE_DESC"),
            "protected": props.get("PROTEGE_4S"),
            "area": props.get("NOM_ARR_VILLE_DESC"),
            "year_round": props.get("SAISONS4"),
            "length_m": props.get("LONGUEUR"),
            "ID_TRC": props.get("ID_TRC")
        }
    }

    clean_features.append(translated_feature)

english_geojson = {
    "type": "FeatureCollection",
    "features": clean_features
}

with open("../data/bike_lanes_clean_english.geojson", "w") as f:
    json.dump(english_geojson, f)




print("Original features:", len(raw["features"]))
print("Dropped outside Montreal:", dropped_outside_city)
print("Dropped for missing geometry or required fields:", dropped_incomplete)
print("Cleaned English features:", len(english_geojson["features"]))
print("Features with missing ID_TRC:", sum(feature["properties"]["ID_TRC"] is None for feature in clean_features))
print("Features with ID_TRC = 0:", sum(feature["properties"]["ID_TRC"] == 0 for feature in clean_features))

Original features: 9485
Dropped outside Montreal: 1238
Dropped for missing geometry or required fields: 2
Cleaned English features: 8245
Features with missing ID_TRC: 3
Features with ID_TRC = 0: 682


3. Read the written file back

The next cell opens the cleaned GeoJSON and prints the number of cleaned features and an example's properties

In [3]:
import json

with open("../data/bike_lanes_clean_english.geojson") as f:
    cleaned = json.load(f)

print(len(cleaned["features"]))
print(cleaned["features"][0]["properties"])

8245
{'facility_type': 'Piste cyclable sur rue', 'protected': 'Oui', 'area': 'Ahuntsic-Cartierville', 'year_round': 'Oui', 'length_m': 71, 'ID_TRC': 4007876.0}


4. Check the exported GeoJSON

Read the file back from disk, as the map will. Check its feature count, geometry types, approximate Montreal coordinate extent, and the English property names needed by the site. The notebook does not check for duplicated segments or perform a boundary check to prove that every route is within Montreal City Limits.

In [4]:
with open("../data/bike_lanes_clean_english.geojson", encoding="utf-8") as f:
    checked = json.load(f)

features = checked["features"]

# Check the file type and feature count
assert checked["type"] == "FeatureCollection"
assert len(features) == len(clean_features), "Feature count does not match"

# Check that every feature has line geometry and the properties the map needs
required_properties = {
    "facility_type",
    "protected",
    "area",
    "year_round",
    "length_m",
    "ID_TRC",
}

assert all(
    feature["geometry"]["type"] in {"LineString", "MultiLineString"}
    for feature in features
), "A feature is not a line"

assert all(
    required_properties <= set(feature["properties"])
    for feature in features
), "A feature is missing a required property"

# Check that the coordinates are in the Montreal area
positions = []
for feature in features:
    geometry = feature["geometry"]
    lines = (
        [geometry["coordinates"]]
        if geometry["type"] == "LineString"
        else geometry["coordinates"]
    )
    for line in lines:
        positions.extend(line)

longitudes = [position[0] for position in positions]
latitudes = [position[1] for position in positions]

assert all(-74.1 < lon < -73.3 for lon in longitudes), "Longitude check failed"
assert all(45.2 < lat < 46.0 for lat in latitudes), "Latitude check failed"

print("All checks passed.")
print("Features:", len(features))
print("Example properties:", features[0]["properties"])

All checks passed.
Features: 8245
Example properties: {'facility_type': 'Piste cyclable sur rue', 'protected': 'Oui', 'area': 'Ahuntsic-Cartierville', 'year_round': 'Oui', 'length_m': 71, 'ID_TRC': 4007876.0}


5. Read Pavement data



In [5]:
import pandas as pd
import json

with open("../data/bike_lanes_clean_english.geojson") as f:
    bike_data = json.load(f)

bike_ids = pd.Series(pd.to_numeric(
    [feature["properties"].get("ID_TRC") for feature in bike_data["features"]],
    errors="coerce"
))

pavement = pd.read_csv("../data/Pavement_data.csv")
print(pavement.columns.tolist())

['ID_TRC', 'Rue', 'De', 'A', 'Longueur', 'Arrondissement', 'DateReleve', 'Indice_PCI', 'Etat_PCI', 'Indice_IRI', 'Etat_IRI']


Find column values and 

In [6]:
print(pavement.columns.tolist())
print(pavement.head())

['ID_TRC', 'Rue', 'De', 'A', 'Longueur', 'Arrondissement', 'DateReleve', 'Indice_PCI', 'Etat_PCI', 'Indice_IRI', 'Etat_IRI']
    ID_TRC                  Rue  ...  Indice_IRI Etat_IRI
0  1160132          Chabot, rue  ...        4.53    Moyen
1  1160133          Chabot, rue  ...        4.66    Moyen
2  1160380  Rosemont, boulevard  ...        2.95    Moyen
3  1160381  Rosemont, boulevard  ...        2.83    Moyen
4  1160382  Rosemont, boulevard  ...        4.12    Moyen

[5 rows x 11 columns]


Check pavement matches 

The pavement table has no missing, zero, or duplicate ID_TRC values, so treat zero and missing bike-lane IDs as unmatchable for this join. Keep these bike features in the main bike network; exclude them only from the pavement overlay. A missing match means the two files could not be linked by ID_TRC, not that the street has no pavement condition.

This check found 5,245 pavement matches among 7,560 bike features with nonzero ID_TRC values. The remaining 2,315 nonzero IDs did not match the pavement table.

In [6]:
pavement["ID_TRC"] = pd.to_numeric(pavement["ID_TRC"], errors="coerce")

bike_ids = pd.Series(pd.to_numeric(
    [feature["properties"].get("ID_TRC") for feature in bike_data["features"]],
    errors="coerce"
))

print("Pavement rows with ID_TRC = 0:", (pavement["ID_TRC"] == 0).sum())
print("Pavement rows missing ID_TRC:", pavement["ID_TRC"].isna().sum())
print("Duplicate pavement IDs:", pavement["ID_TRC"].duplicated().sum())
print("Bike features with ID_TRC = 0:", (bike_ids == 0).sum())
print("Bike features missing ID_TRC:", bike_ids.isna().sum())

usable_bike_ids = bike_ids[bike_ids.notna() & (bike_ids != 0)]
pavement_ids = set(pavement["ID_TRC"].dropna())

print(
    "Bike features with a pavement match:",
    usable_bike_ids.isin(pavement_ids).sum(),
    "of",
    len(usable_bike_ids)
)

Pavement rows with ID_TRC = 0: 0
Pavement rows missing ID_TRC: 0
Duplicate pavement IDs: 0
Bike features with ID_TRC = 0: 682
Bike features missing ID_TRC: 3
Bike features with a pavement match: 5245 of 7560


Join pavement data to bike-network segments

I joined the pavement CSV to the bike-network GeoJSON using the shared ID_TRC street-segment identifer. The pavement table has no missing, zero, or duplicate IDs. For this pavement overlay, I exclude bike features with a missing or zero ID_TRC, and kept only features with a match in the pavement table. This does not remove unmatched routes from the main bike network: it only means they are not included in the pavement overlay. No match means pavement data could not be linked by this ID; it does not mean the street has no pavement condition. PCI and IRI are pavement measurements for matched street segments, not necessarily measurements specific to the bike-lane surface.

In [7]:
import json
import pandas as pd

# Load the cleaned bike GeoJSON and pavement table
with open("../data/bike_lanes_clean_english.geojson", encoding="utf-8") as f:
    bike_data = json.load(f)

pavement = pd.read_csv("../data/Pavement_data.csv")

# Convert both join columns to numbers, then to nullable integers
pavement["ID_TRC"] = pd.to_numeric(pavement["ID_TRC"], errors="coerce").astype("Int64")

bike_rows = pd.DataFrame({
    "feature_index": range(len(bike_data["features"])),
    "ID_TRC": [
        feature["properties"].get("ID_TRC")
        for feature in bike_data["features"]
    ]
})
bike_rows["ID_TRC"] = pd.to_numeric(
    bike_rows["ID_TRC"], errors="coerce"
).astype("Int64")

# Exclude missing or zero IDs from this join only
eligible_bikes = bike_rows[
    bike_rows["ID_TRC"].notna() & (bike_rows["ID_TRC"] != 0)
].copy()

# Keep the pavement fields this overlay will use
pavement_fields = pavement[
    ["ID_TRC", "Indice_PCI", "Etat_PCI", "Indice_IRI", "Etat_IRI", "DateReleve"]
].dropna(subset=["ID_TRC"])

# Join bike features to pavement rows; requires pavement IDs to be unique
matched = eligible_bikes.merge(
    pavement_fields,
    on="ID_TRC",
    how="inner",
    validate="many_to_one"
)

# Build a separate GeoJSON, preserving the matched bike geometries
pavement_features = []
field_names = {
    "pavement_pci": "Indice_PCI",
    "pavement_condition": "Etat_PCI",
    "roughness_iri": "Indice_IRI",
    "roughness_condition": "Etat_IRI",
    "survey_date": "DateReleve"
}

for _, row in matched.iterrows():
    original = bike_data["features"][int(row["feature_index"])]
    properties = dict(original["properties"])

    for output_name, source_name in field_names.items():
        value = row[source_name]
        if pd.isna(value):
            properties[output_name] = None
        elif hasattr(value, "item"):
            properties[output_name] = value.item()
        else:
            properties[output_name] = value

    pavement_features.append({
        "type": "Feature",
        "geometry": original["geometry"],
        "properties": properties
    })

pavement_geojson = {
    "type": "FeatureCollection",
    "features": pavement_features
}

output_path = "../data/bike_lanes_with_pavement.geojson"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(pavement_geojson, f, ensure_ascii=False)

print("Bike features eligible for matching:", len(eligible_bikes))
print("Bike features matched to pavement:", len(matched))
print("Eligible bike features without a pavement match:", len(eligible_bikes) - len(matched))
print("Wrote:", output_path)

Bike features eligible for matching: 7560
Bike features matched to pavement: 5245
Eligible bike features without a pavement match: 2315
Wrote: ../data/bike_lanes_with_pavement.geojson


Checkkkkk

In [8]:
import geopandas as gpd
import os

FILE = "../data/bike_lanes_with_pavement.geojson"
EXPECTED = 5245
POPUP_FIELDS = [
    "pavement_pci",
    "pavement_condition",
    "roughness_iri",
    "roughness_condition",
    "survey_date"
]

back = gpd.read_file(FILE)

assert len(back) == EXPECTED, f"Expected {EXPECTED} features, got {len(back)}"
assert back.crs is not None and back.crs.to_epsg() == 4326, f"Unexpected CRS: {back.crs}"
assert back.geom_type.isin(["LineString", "MultiLineString"]).all(), "A feature is not a line"
assert all(field in back.columns for field in POPUP_FIELDS), "A required pavement field is missing"
assert back["ID_TRC"].notna().all() and (back["ID_TRC"] != 0).all(), "An invalid ID_TRC is present"

minx, miny, maxx, maxy = back.total_bounds
assert -74.1 < minx and maxx < -73.3, "Longitude extent is outside the expected area"
assert 45.2 < miny and maxy < 46.0, "Latitude extent is outside the expected area"

size_mb = os.path.getsize(FILE) / 1_000_000
assert size_mb <= 5, f"File is {size_mb:.2f} MB, over the 5 MB guide"

print("All checks passed")
print("Features:", len(back))
print("Geometry types:", sorted(back.geom_type.unique()))
print("File size:", round(size_mb, 2), "MB")


All checks passed
Features: 5245
Geometry types: ['LineString']
File size: 3.43 MB


In [16]:
print(back["pavement_pci"].describe())
print(back["pavement_condition"].value_counts(dropna=False))

count    5245.000000
mean       66.501049
std        21.319423
min         3.000000
25%        52.000000
50%        70.000000
75%        83.000000
max       100.000000
Name: pavement_pci, dtype: float64
pavement_condition
Moyen           1520
Bon             1358
Excellent       1031
Mauvais          874
Très mauvais     462
Name: count, dtype: int64
